# 04 – Target formulation and ablation (version 2)
Compares price and return targets, rebuilds the price model with lagged prices, and runs the date-matched ablation (split × target × scaling). This is the version behind Tables 4.2–4.4.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import os
import itertools
from datetime import datetime

import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, matthews_corrcoef,
                             mean_squared_error, mean_absolute_error, r2_score)

from google.colab import drive
drive.mount('/content/drive')

BASE     = "/content/drive/MyDrive/CS5500_dissertation"
FEATURES = f"{BASE}/features"
RESULTS  = f"{BASE}/results"
FIGURES  = f"{BASE}/figures"
os.makedirs(FIGURES, exist_ok=True)

TICKERS = ["AAPL", "MSFT", "JPM", "XOM", "SPY"]
SEED    = 42
DEV_END = "2023-12-31"

INITIAL_TRAIN, TEST_SIZE, STEP = 1000, 250, 250

np.random.seed(SEED)

dev = {t: pd.read_csv(f"{FEATURES}/{t}_features.csv",
                      index_col=0, parse_dates=True).loc[:DEV_END]
       for t in TICKERS}

PREDICTORS = [c for c in dev["AAPL"].columns
              if not c.startswith("target") and c != "price_today"]

print(f"Loaded {len(TICKERS)} assets, {len(PREDICTORS)} predictors")

In [ ]:
def expanding_walk_forward(n_rows, initial_train=INITIAL_TRAIN,
                           test_size=TEST_SIZE, step=STEP):
    start = initial_train
    while start + test_size <= n_rows:
        yield np.arange(0, start), np.arange(start, start + test_size)
        start += step


def random_split(n_rows, test_frac=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    p = rng.permutation(n_rows)
    n_test = int(n_rows * test_frac)
    return p[n_test:], p[:n_test]


print("Splitters defined")

## 2. Why the price-level target flatters a model
Autocorrelation of prices versus returns, and the copy-yesterday benchmark.

In [ ]:
df = dev["AAPL"]

price  = df["price_today"]
ret    = df["target_return"]

print("AAPL, development period\n")
print(f"  Price   lag-1 autocorrelation: {price.autocorr(1):.6f}")
print(f"  Return  lag-1 autocorrelation: {ret.autocorr(1):.6f}")
print()
print("  A near-1.0 autocorrelation means the previous value alone explains")
print("  almost all variation. A near-0.0 autocorrelation means it explains")
print("  almost none. These are not the same task.")

# The trivial 'copy today' predictor, scored on the price target
naive_price = price.shift(1).dropna()
actual      = price.loc[naive_price.index]

print(f"\n  Copy-yesterday's-price predictor:")
print(f"    R2:   {r2_score(actual, naive_price):.6f}")
print(f"    RMSE: {np.sqrt(mean_squared_error(actual, naive_price)):.4f}")
print("\n  This model contains no information whatsoever.")

## 3. The two targets, same features, same split
Both arms use walk-forward validation and fold-fitted scaling, so splitting and preprocessing are held constant.

In [ ]:
def run_price_target(df):
    """Linear regression predicting tomorrow's price level.
    Returns regression metrics AND the implied directional call."""
    X = df[PREDICTORS].values
    y = df["target_price"].values
    px_today = df["price_today"].values
    y_dir = df["target_direction"].values

    preds, actuals, dir_pred, dir_true, today = [], [], [], [], []

    for tr, te in expanding_walk_forward(len(df)):
        scaler = StandardScaler().fit(X[tr])
        model  = LinearRegression()
        model.fit(scaler.transform(X[tr]), y[tr])

        p = model.predict(scaler.transform(X[te]))

        preds.extend(p)
        actuals.extend(y[te])
        today.extend(px_today[te])
        # implied direction: is the forecast above today's known close?
        dir_pred.extend((p > px_today[te]).astype(int))
        dir_true.extend(y_dir[te])

    return (np.array(preds), np.array(actuals), np.array(today),
            np.array(dir_pred), np.array(dir_true))


def run_return_target(df):
    """Linear regression predicting tomorrow's return, converted back to an
    implied price so RMSE is measured on the same scale as the price model."""
    X = df[PREDICTORS].values
    y = df["target_return"].values
    px_today = df["price_today"].values
    y_price  = df["target_price"].values
    y_dir    = df["target_direction"].values

    preds, actuals, dir_pred, dir_true = [], [], [], []

    for tr, te in expanding_walk_forward(len(df)):
        scaler = StandardScaler().fit(X[tr])
        model  = LinearRegression()
        model.fit(scaler.transform(X[tr]), y[tr])

        r = model.predict(scaler.transform(X[te]))

        # implied price uses today's close, which is known at forecast time
        preds.extend(px_today[te] * (1 + r))
        actuals.extend(y_price[te])
        dir_pred.extend((r > 0).astype(int))
        dir_true.extend(y_dir[te])

    return (np.array(preds), np.array(actuals),
            np.array(dir_pred), np.array(dir_true))


rows = []

for t in TICKERS:
    d = dev[t]

    p_pred, p_act, p_today, p_dir, p_dtrue = run_price_target(d)
    r_pred, r_act, r_dir, r_dtrue          = run_return_target(d)

    # the do-nothing benchmark on the price target
    naive_rmse = np.sqrt(mean_squared_error(p_act, p_today))

    rows.append({
        "ticker": t, "target": "price_level",
        "rmse":  round(np.sqrt(mean_squared_error(p_act, p_pred)), 4),
        "mae":   round(mean_absolute_error(p_act, p_pred), 4),
        "r2":    round(r2_score(p_act, p_pred), 6),
        "dir_acc": round(accuracy_score(p_dtrue, p_dir), 4),
        "mcc":   round(matthews_corrcoef(p_dtrue, p_dir), 4),
    })

    rows.append({
        "ticker": t, "target": "return",
        "rmse":  round(np.sqrt(mean_squared_error(r_act, r_pred)), 4),
        "mae":   round(mean_absolute_error(r_act, r_pred), 4),
        "r2":    round(r2_score(r_act, r_pred), 6),
        "dir_acc": round(accuracy_score(r_dtrue, r_dir), 4),
        "mcc":   round(matthews_corrcoef(r_dtrue, r_dir), 4),
    })

    rows.append({
        "ticker": t, "target": "naive_copy_today",
        "rmse":  round(naive_rmse, 4),
        "mae":   round(mean_absolute_error(p_act, p_today), 4),
        "r2":    round(r2_score(p_act, p_today), 6),
        "dir_acc": np.nan,
        "mcc":   np.nan,
    })

target_results = pd.DataFrame(rows)
print(target_results.to_string(index=False))

target_results.to_csv(f"{RESULTS}/target_formulation.csv", index=False)

## 3b. Correcting the price-target arm
The feature set built in notebook 02 is deliberately stationary: returns, ratios, volatility, momentum indicators.

In [ ]:
# The price-target arm must include lagged PRICES as inputs, because that is
# what the models being audited actually use. Without them the model has no
# information about price level and cannot perform the task at all.

def run_price_target_fixed(df, n_lags=5):
    d = df.copy()
    for lag in range(n_lags):
        d[f"px_lag_{lag}"] = d["price_today"].shift(lag)
    d = d.dropna()

    cols     = PREDICTORS + [f"px_lag_{i}" for i in range(n_lags)]
    X        = d[cols].values
    y        = d["target_price"].values
    px_today = d["price_today"].values
    y_dir    = d["target_direction"].values

    pred, act, today, dp, dt = [], [], [], [], []

    for tr, te in expanding_walk_forward(len(d)):
        sc = StandardScaler().fit(X[tr])
        m  = LinearRegression().fit(sc.transform(X[tr]), y[tr])
        p  = m.predict(sc.transform(X[te]))

        pred.extend(p); act.extend(y[te]); today.extend(px_today[te])
        dp.extend((p > px_today[te]).astype(int)); dt.extend(y_dir[te])

    return (np.array(pred), np.array(act), np.array(today),
            np.array(dp), np.array(dt))


rows_fixed = []
for t in TICKERS:
    p, a, tod, dp, dt = run_price_target_fixed(dev[t])
    rows_fixed.append({
        "ticker": t,
        "r2":         round(r2_score(a, p), 6),
        "rmse":       round(np.sqrt(mean_squared_error(a, p)), 4),
        "naive_rmse": round(np.sqrt(mean_squared_error(a, tod)), 4),
        "beats_naive": np.sqrt(mean_squared_error(a, p)) < np.sqrt(mean_squared_error(a, tod)),
        "dir_acc":    round(accuracy_score(dt, dp), 4),
        "mcc":        round(matthews_corrcoef(dt, dp), 4),
    })

price_fixed = pd.DataFrame(rows_fixed)
print("Price-target model with lagged prices as inputs\n")
print(price_fixed.to_string(index=False))
print(f"\nMean R2: {price_fixed.r2.mean():.6f}")
print(f"Mean directional accuracy: {price_fixed.dir_acc.mean():.4f}")
print(f"Beats naive on RMSE: {int(price_fixed.beats_naive.sum())}/{len(price_fixed)}")

price_fixed.to_csv(f"{RESULTS}/price_target_fixed.csv", index=False)

## 4. The headline comparison
Read the R² column against the directional accuracy column.

In [ ]:
price_arm = target_results.query("target == 'price_level'").set_index("ticker")
naive_arm = target_results.query("target == 'naive_copy_today'").set_index("ticker")

summary = pd.DataFrame({
    "model_r2":       price_arm["r2"],
    "model_rmse":     price_arm["rmse"],
    "naive_rmse":     naive_arm["rmse"],
    "beats_naive":    price_arm["rmse"] < naive_arm["rmse"],
    "dir_accuracy":   price_arm["dir_acc"],
    "mcc":            price_arm["mcc"],
})

print("Price-level model: impressive by regression metrics, useless directionally\n")
print(summary.to_string())

print(f"\nMean R2 across assets:            {summary.model_r2.mean():.4f}")
print(f"Mean directional accuracy:        {summary.dir_accuracy.mean():.4f}")
print(f"Mean MCC:                         {summary.mcc.mean():.4f}")
print(f"Assets where model beats naive:   {int(summary.beats_naive.sum())}/{len(summary)}")

## 5. The figure
Figure 4.1: the price predictions look almost perfect, but direction is close to chance.

In [ ]:
p_pred, p_act, p_today, p_dir, p_dtrue = run_price_target_fixed(dev["AAPL"])

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Panel 1: the full test year - the chart that appears in the literature
window = slice(-250, None)
axes[0].plot(p_act[window],   label="actual",         linewidth=1.2)
axes[0].plot(p_pred[window],  label="predicted",      linewidth=1.2, alpha=0.8)
axes[0].plot(p_today[window], label="copy yesterday", linewidth=1.0, linestyle=":")
axes[0].set_title(f"Full test year (R2 = {r2_score(p_act, p_pred):.4f})")
axes[0].set_xlabel("trading day"); axes[0].set_ylabel("price"); axes[0].legend()

# Panel 2: 30-day detail, where the one-day lag becomes visible
zoom = slice(-40, -10)
axes[1].plot(p_act[zoom],   label="actual",    linewidth=1.8, marker="o", markersize=3)
axes[1].plot(p_pred[zoom],  label="predicted", linewidth=1.8, marker="s", markersize=3)
axes[1].plot(p_today[zoom], label="copy yesterday", linewidth=1.4, linestyle=":")
axes[1].set_title("Zoomed: 30 days")
axes[1].set_xlabel("trading day"); axes[1].legend()

# Panel 3: what those same predictions achieve directionally
acc = accuracy_score(p_dtrue, p_dir)
axes[2].bar(["correct", "incorrect"], [acc, 1 - acc], color=["#4a7c59", "#a44a3f"])
axes[2].axhline(0.5, linestyle="--", color="black", linewidth=1)
axes[2].set_ylim(0, 1)
axes[2].set_title(f"Directional accuracy = {acc:.4f}")
axes[2].set_ylabel("proportion")

plt.tight_layout()
plt.savefig(f"{FIGURES}/price_vs_direction.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved to {FIGURES}/price_vs_direction.png")

## 6. Full single-factor ablation (RQ3)
Eight configurations: every combination of the three protocol choices.

In [ ]:
def get_wf_folds(df):
    """All walk-forward folds as (train_end, test_end) row positions."""
    folds, start = [], INITIAL_TRAIN
    while start + TEST_SIZE <= len(df):
        folds.append((start, start + TEST_SIZE))
        start += STEP
    return folds


def ablation_run(df, price_target, random_sp, global_scale):
    """One cell of the ablation grid.

    The random-split arm is restricted to the same rows the walk-forward arm
    tests on, so protocol effects are not confounded with differences in test
    period or sample size. Without this correction the earlier neural
    comparison reversed the sign of the measured effect.
    """
    X        = df[PREDICTORS].values
    px_today = df["price_today"].values
    y_dir    = df["target_direction"].values
    y        = df["target_price"].values if price_target else df["target_return"].values

    folds = get_wf_folds(df)
    first_test, last_test = folds[0][0], folds[-1][1]

    X_use = StandardScaler().fit_transform(X) if global_scale else X

    if random_sp:
        eligible = np.arange(first_test, last_test)      # matched test window
        rng = np.random.default_rng(SEED)
        te  = rng.permutation(eligible)
        tr  = np.setdiff1d(np.arange(len(df)), te)
        splits = [(tr, te)]
    else:
        splits = [(np.arange(0, s), np.arange(s, e)) for s, e in folds]

    dir_pred, dir_true = [], []

    for tr, te in splits:
        if global_scale:
            Xtr, Xte = X_use[tr], X_use[te]
        else:
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])

        model = LinearRegression()
        model.fit(Xtr, y[tr])
        p = model.predict(Xte)

        if price_target:
            dir_pred.extend((p > px_today[te]).astype(int))
        else:
            dir_pred.extend((p > 0).astype(int))

        dir_true.extend(y_dir[te])

    return np.array(dir_true), np.array(dir_pred)


print("ablation_run defined - random arm date-matched to walk-forward folds")

In [ ]:
grid = []

for t in TICKERS:
    for price_t, rand_s, glob_sc in itertools.product([True, False], repeat=3):
        yt, yp = ablation_run(dev[t], price_t, rand_s, glob_sc)

        grid.append({
            "ticker":   t,
            "target":   "price"  if price_t else "return",
            "split":    "random" if rand_s  else "walkfwd",
            "scaling":  "global" if glob_sc else "fold",
            "n":        len(yt),
            "accuracy": round(accuracy_score(yt, yp), 4),
            "mcc":      round(matthews_corrcoef(yt, yp), 4),
        })

ablation = pd.DataFrame(grid)

print("SAMPLE SIZE CHECK - both split types must now match\n")
print(ablation.groupby("split")["n"].agg(["min", "max", "mean"]).to_string())
matched = ablation.groupby("split")["n"].nunique().max() == 1 and \
          ablation["n"].nunique() == 1
print(f"\nMatched: {matched}")

print("\n\nAblation grid, mean across five assets\n")
print(ablation.groupby(["target", "split", "scaling"])[["accuracy", "mcc"]]
              .mean().round(4).to_string())

ablation.to_csv(f"{RESULTS}/ablation_grid.csv", index=False)

## 7. Attribution: main effect of each factor
Averaging over the other two factors isolates each one's individual contribution.

In [ ]:
effects = []

for factor, leak_level, sound_level in [
    ("target",  "price",  "return"),
    ("split",   "random", "walkfwd"),
    ("scaling", "global", "fold"),
]:
    leak  = ablation.query(f"{factor} == '{leak_level}'")["accuracy"].mean()
    sound = ablation.query(f"{factor} == '{sound_level}'")["accuracy"].mean()

    effects.append({
        "factor":            factor,
        "leakage_prone":     round(leak, 4),
        "sound":             round(sound, 4),
        "effect_pct_points": round(100 * (leak - sound), 3),
    })

attribution = pd.DataFrame(effects).sort_values(
    "effect_pct_points", key=abs, ascending=False
)

print("Main effect of each protocol choice on directional accuracy")
print("(positive = the leakage-prone option inflates the result)\n")
print(attribution.to_string(index=False))

full_leak = ablation.query("target=='price' and split=='random' and scaling=='global'")
full_sound = ablation.query("target=='return' and split=='walkfwd' and scaling=='fold'")

print(f"\nFully leakage-prone configuration: {full_leak.accuracy.mean():.4f}")
print(f"Fully sound configuration:         {full_sound.accuracy.mean():.4f}")
print(f"Total gap:                         "
      f"{100*(full_leak.accuracy.mean() - full_sound.accuracy.mean()):.2f} percentage points")

attribution.to_csv(f"{RESULTS}/attribution.csv", index=False)

## 8. Append to the results log

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

lines = []
lines.append("")
lines.append("---")
lines.append("")
lines.append(f"## {today} - Notebook 04: target formulation and ablation")
lines.append("")
lines.append("**Target formulation, walk-forward split and fold scaling held constant**")
lines.append("")
lines.append(target_results.to_string(index=False))
lines.append("")
lines.append(f"- Mean R2 of the price-level model: {summary.model_r2.mean():.4f}")
lines.append(f"- Mean directional accuracy of that same model: {summary.dir_accuracy.mean():.4f}")
lines.append(f"- Mean MCC of that same model: {summary.mcc.mean():.4f}")
lines.append(f"- Assets where the model's RMSE beats copy-today: "
             f"{int(summary.beats_naive.sum())} of {len(summary)}")
lines.append("")
lines.append("**Full ablation grid, mean across assets**")
lines.append("")
lines.append(ablation.groupby(["target","split","scaling"])[["accuracy","mcc"]]
                     .mean().round(4).to_string())
lines.append("")
lines.append("**Attribution: main effect of each factor**")
lines.append("")
lines.append(attribution.to_string(index=False))
lines.append("")
lines.append(f"- Fully leakage-prone: {full_leak.accuracy.mean():.4f}")
lines.append(f"- Fully sound: {full_sound.accuracy.mean():.4f}")
lines.append(f"- Total gap: {100*(full_leak.accuracy.mean()-full_sound.accuracy.mean()):.2f} pts")
lines.append("")
lines.append("**Note:** linear models only at this stage. The capacity hypothesis "
             "(that leakage inflates flexible models more than rigid ones) is tested "
             "in notebook 05 with LSTM, GRU and CNN-LSTM.")
lines.append("")
lines.append("**Next:** Notebook 05 - neural architectures under both protocols")

entry = "\n".join(lines)

with open(f"{BASE}/results_log.md", "a") as f:
    f.write(entry)

print(entry)